# Proportional diversion rule

**What this notebook does.** It implements the proportional diversion rule from the proposal and compares it with the current pairwise formula (`shockt_engine.diversion_rate`) on Cases 1 and 2.

**The current formula** attributes the full gain of a "to" origin to a single named "from" origin:
`r = (s_to,response − s_to,baseline) / s_from,baseline`.
For servers, Mexico → Taiwan is 0.31.

**The proportional rule** distributes a gainer's gain across *all* losers in proportion to each loser's loss:
`r_B→A = gain_A × (loss_B / total_loss) / s_B,baseline`.
For servers, Mexico → Taiwan becomes ~0.28 because other origins also lost share.

**To run:** set `ANALYST_DATABASE_URL` in `analysis/.env`, then run all cells. It reads; it never writes to the database.

## 1. Setup

Load the same supply data used by the Case 1 and Case 2 notebooks, and their shock windows.

In [ ]:

import matplotlib.pyplot as plt
import polars as pl
from shockt_engine import (
    diversion_rate,
    shares,
    window_summary,
)

from shockt_analysis import case2, cases, core
from shockt_analysis.settings import REPO_ROOT, get_settings

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_width_chars(160)
pl.Config.set_float_precision(4)

CASE1_DIR = REPO_ROOT / "inputs" / "cases" / "tariffs_2025"
CASE2_DIR = REPO_ROOT / "inputs" / "cases" / "panama_red_sea_2023"

SERVERS = ["847150"]
GEAR = ["850421", "850422", "850423", "850433", "850434", "853720"]
SERVER_ORIGINS = ["TWN", "MEX", "THA", "VNM"]
GEAR_ORIGINS = ["MEX", "KOR", "CHN", "TWN", "THA", "VNM"]

case1_windows = cases.load_windows(CASE1_DIR / "shock_windows.yaml")
[(w.name, w.first, w.last) for w in case1_windows]

In [ ]:
with core.connect(get_settings()) as conn:
    supply = core.supply(conn, SERVERS + GEAR)
    transits = core.chokepoint_transits(conn)
    loads = core.current_loads(
        conn,
        ["census_hs", "census_porths", "census_statehs", "bts_transborder",
         "portwatch_chokepoints"],
    )

case2_case_windows = case2.load_case_windows(CASE2_DIR / "shock_windows.yaml")
case2_shock_windows, _ = case2.shock_windows(transits, case2_case_windows)
case2_windows = [case2_case_windows.baseline, *case2_shock_windows]

print(f"Supply rows: {supply.height:,}; months {supply['month'].min()} to {supply['month'].max()}")
loads

## 2. Current pairwise rates

Reproduce Case 1 Section 4: each origin's share of server tonnage per window, the share changes, and the diversion rate from Mexico to Taiwan. The pairwise formula attributes the full gain of Taiwan's share to Mexico:
`r = (s_TWN,response − s_TWN,baseline) / s_MEX,baseline`.

In [ ]:
servers_monthly = cases.monthly_supply(supply, SERVERS, SERVER_ORIGINS)
origin_shares = cases.window_shares(servers_monthly, case1_windows, ["origin_group"])
shares_table = origin_shares.pivot(
    on="window", index="origin_group", values="share"
).select("origin_group", *[w.name for w in case1_windows])
changes = cases.share_changes(origin_shares, "baseline", "response", "origin_group")
shares_table.with_columns(
    pl.col("origin_group")
    .replace_strict(changes, default=None)
    .alias("change_baseline_to_response")
)

In [ ]:
def monthly_origin_shares(monthly):
    totals = monthly.group_by("month", "origin_group").agg(pl.col("weight_t").sum())
    return {
        m: shares(
            dict(zip(part["origin_group"].to_list(), part["weight_t"].to_list(), strict=True))
        )
        for (m,), part in totals.group_by("month")
    }


def diversion_by_window(monthly, source, target, windows, origin_shares_df):
    base = origin_shares_df.filter(pl.col("window") == "baseline")
    baseline = dict(zip(base["origin_group"].to_list(), base["share"].to_list(), strict=True))
    by_month = monthly_origin_shares(monthly)
    rates = {
        m: diversion_rate(baseline[source], baseline[target], s.get(target, 0.0))
        for m, s in by_month.items()
    }
    tonnes = dict(monthly.group_by("month").agg(pl.col("weight_t").sum()).iter_rows())
    later = [w for w in windows if w.name != "baseline"]
    return {name: s for name, s in window_summary(rates, later, tonnes).items()}


mex_to_twn = diversion_by_window(
    servers_monthly, "MEX", "TWN", case1_windows, origin_shares
)
print("Pairwise diversion rate, MEX → TWN:")
pl.DataFrame(
    [{"window": k, "mean": v.mean, "low": v.low, "high": v.high, "months": v.months}
     for k, v in mex_to_twn.items()]
)

## 3. The proportional formula

When multiple origins lose share while one gains, the pairwise formula overstates the attribution to any single loser. The proportional rule distributes a gainer's gain across all losers in proportion to each loser's share of the total loss:

```
loss_B      = max(0, s_B,baseline − s_B,response)
total_loss  = Σ loss over all losers
r_B→A       = gain_A × (loss_B / total_loss) / s_B,baseline
```

The constraint is that the attributed gains from all losers to a gainer sum to the actual gain: `Σ_B (r_B→A × s_B,baseline) = gain_A`. The proportional weights `loss_B / total_loss` satisfy this.

In [ ]:
def proportional_diversion_rates(
    baseline_shares: dict[str, float],
    response_shares: dict[str, float],
) -> dict[tuple[str, str], float]:
    """For each (loser, gainer) pair, the proportional diversion rate."""
    all_origins = set(baseline_shares) | set(response_shares)
    losses = {
        o: max(0.0, baseline_shares.get(o, 0.0) - response_shares.get(o, 0.0))
        for o in all_origins
    }
    gains = {
        o: max(0.0, response_shares.get(o, 0.0) - baseline_shares.get(o, 0.0))
        for o in all_origins
    }
    total_loss = sum(losses.values())
    rates: dict[tuple[str, str], float] = {}
    if total_loss <= 0:
        return rates
    for gainer, gain in gains.items():
        if gain <= 0:
            continue
        for loser, loss in losses.items():
            if loss <= 0 or baseline_shares.get(loser, 0.0) <= 0:
                continue
            rates[(loser, gainer)] = (
                gain * (loss / total_loss) / baseline_shares[loser]
            )
    return rates


def pairwise_diversion_rates(
    baseline_shares: dict[str, float],
    response_shares: dict[str, float],
) -> dict[tuple[str, str], float]:
    """For each (loser, gainer) pair, the current pairwise rate."""
    all_origins = set(baseline_shares) | set(response_shares)
    rates: dict[tuple[str, str], float] = {}
    for loser in all_origins:
        s_from = baseline_shares.get(loser, 0.0)
        if s_from <= 0 or response_shares.get(loser, 0.0) >= s_from:
            continue
        for gainer in all_origins:
            s_to_b = baseline_shares.get(gainer, 0.0)
            s_to_r = response_shares.get(gainer, 0.0)
            if s_to_r <= s_to_b:
                continue
            rates[(loser, gainer)] = diversion_rate(s_from, s_to_b, s_to_r)
    return rates


# Quick verification: the proportional rates' attributed gains sum to the actual gain.
base_s = dict(
    origin_shares.filter(pl.col("window") == "baseline")
    .select("origin_group", "share")
    .iter_rows()
)
resp_s = dict(
    origin_shares.filter(pl.col("window") == "response")
    .select("origin_group", "share")
    .iter_rows()
)
prop_rates = proportional_diversion_rates(base_s, resp_s)
for gainer in {g for _, g in prop_rates}:
    attributed = sum(
        r * base_s[loser]
        for (loser, g), r in prop_rates.items()
        if g == gainer
    )
    actual_gain = resp_s.get(gainer, 0.0) - base_s.get(gainer, 0.0)
    print(f"{gainer}: attributed = {attributed:.6f}, actual gain = {actual_gain:.6f}")

## 4. Proportional rates for servers

The same pairs as the Case 1 notebook, now with the proportional rule. Mexico → Taiwan should drop from ~0.31 to ~0.28 because Vietnam and the `other` group also lost share.

In [ ]:
pair_rates = pairwise_diversion_rates(base_s, resp_s)

comparison_rows = []
for (loser, gainer), r_prop in sorted(prop_rates.items()):
    r_pair = pair_rates.get((loser, gainer))
    comparison_rows.append({
        "from": loser,
        "to": gainer,
        "pairwise_rate": r_pair,
        "proportional_rate": r_prop,
        "difference": r_prop - r_pair if r_pair is not None else None,
    })

comparison = pl.DataFrame(comparison_rows)
print("Server diversion rates, baseline → response:")
comparison

In [ ]:
# Highlight the key pair
mex_twn_pair = pair_rates.get(("MEX", "TWN"), 0.0)
mex_twn_prop = prop_rates.get(("MEX", "TWN"), 0.0)
print(f"MEX → TWN pairwise:      {mex_twn_pair:.4f}")
print(f"MEX → TWN proportional:  {mex_twn_prop:.4f}")
print(f"Difference:              {mex_twn_prop - mex_twn_pair:+.4f}")

## 5. All-pairs diversion matrix

A full origin × origin matrix for servers. The left half is the pairwise formula (each cell independent), the right half the proportional rule (cells in each column sum to the actual gain). Each row is a loser; each column is a gainer.

In [ ]:
origins = sorted(base_s.keys())
losers = [o for o in origins if base_s[o] > resp_s.get(o, 0.0)]
gainers = [o for o in origins if resp_s.get(o, 0.0) > base_s.get(o, 0.0)]


def matrix_frame(rates, label):
    rows = []
    for loser in losers:
        row = {"from (loser)": loser}
        for gainer in gainers:
            row[gainer] = rates.get((loser, gainer))
        rows.append(row)
    return pl.DataFrame(rows)


print("Pairwise formula:")
display(matrix_frame(pair_rates, "pairwise"))

print("\nProportional formula:")
display(matrix_frame(prop_rates, "proportional"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, rates, title in [
    (axes[0], pair_rates, "Pairwise"),
    (axes[1], prop_rates, "Proportional"),
]:
    width = 0.8 / len(gainers)
    for j, gainer in enumerate(gainers):
        vals = [rates.get((loser, gainer), 0.0) for loser in losers]
        ax.bar(
            [i + j * width for i in range(len(losers))],
            vals,
            width,
            label=f"→ {gainer}",
        )
    ax.set_xticks(
        [i + width * (len(gainers) - 1) / 2 for i in range(len(losers))], losers
    )
    ax.set_title(title)
    ax.set_ylabel("diversion rate")
    ax.legend(fontsize=7, loc="upper right")

fig.suptitle("Server diversion rates, baseline → response", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Apply to Case 2 gear

The same comparison for the electrical gear's Asian origin shares, using the Panama Canal drought and Red Sea diversion windows from the Case 2 notebook. These windows are computed from PortWatch chokepoint transits, not hardcoded.

In [ ]:
gear_monthly = cases.monthly_supply(supply, GEAR, GEAR_ORIGINS)
gear_origin_shares = cases.window_shares(gear_monthly, case2_windows, ["origin_group"])

gear_shares_table = gear_origin_shares.pivot(
    on="window", index="origin_group", values="share"
).select("origin_group", *[w.name for w in case2_windows])
gear_shares_table

In [ ]:
gear_results = []
for shock in case2_shock_windows:
    gear_base = dict(
        gear_origin_shares.filter(pl.col("window") == "baseline")
        .select("origin_group", "share").iter_rows()
    )
    gear_resp = dict(
        gear_origin_shares.filter(pl.col("window") == shock.name)
        .select("origin_group", "share").iter_rows()
    )
    gear_pair = pairwise_diversion_rates(gear_base, gear_resp)
    gear_prop = proportional_diversion_rates(gear_base, gear_resp)

    for (loser, gainer), r_prop in sorted(gear_prop.items()):
        r_pair = gear_pair.get((loser, gainer))
        gear_results.append({
            "window": shock.name,
            "from": loser,
            "to": gainer,
            "pairwise_rate": r_pair,
            "proportional_rate": r_prop,
            "difference": r_prop - r_pair if r_pair is not None else None,
        })

gear_comparison = pl.DataFrame(gear_results)
print("Gear diversion rates by shock window:")
gear_comparison

In [ ]:
# Summary: largest differences between the two formulas for gear
top_diffs = (
    gear_comparison.filter(pl.col("difference").is_not_null())
    .sort(pl.col("difference").abs(), descending=True)
    .head(10)
)
print("Gear: largest differences between pairwise and proportional rates:")
top_diffs

## 7. What changes

Compare the pairwise and proportional rates side by side. The proportional rule changes the numbers but should not change the qualitative findings: who gained, who lost, and which shifts are large enough to matter.

In [ ]:
# For servers: the headline pair and the difference
print("=== SERVERS (Case 1, baseline → response) ===")
print(f"  Pairwise  MEX → TWN: {mex_twn_pair:.4f}")
print(f"  Proportional MEX → TWN: {mex_twn_prop:.4f}")
print(f"  The proportional rate is {(mex_twn_prop / mex_twn_pair - 1) * 100:+.1f}% "
      f"of the pairwise rate.")
print()

# The constraint check: attributed gains sum to actual gain
for label, rates_dict, base, resp in [
    ("servers (pairwise)", pair_rates, base_s, resp_s),
    ("servers (proportional)", prop_rates, base_s, resp_s),
]:
    for gainer in {g for _, g in rates_dict}:
        attributed = sum(
            r * base[loser] for (loser, g), r in rates_dict.items() if g == gainer
        )
        actual = resp.get(gainer, 0.0) - base.get(gainer, 0.0)
        print(f"  {label}, {gainer}: attributed {attributed:.4f}, actual gain {actual:.4f}")

print()
print("The pairwise formula over-attributes (each loser gets the full gain),")
print("the proportional formula sums to the actual gain by construction.")

In [ ]:
# Summary chart: key pairs, both formulas
key_pairs = [
    ("servers", "MEX", "TWN", base_s, resp_s),
    ("servers", "MEX", "THA", base_s, resp_s),
]
# Add gear pairs for the Red Sea window
red_sea_name = [w.name for w in case2_shock_windows if "red_sea" in w.name]
if red_sea_name:
    gear_base_rs = dict(
        gear_origin_shares.filter(pl.col("window") == "baseline")
        .select("origin_group", "share").iter_rows()
    )
    gear_resp_rs = dict(
        gear_origin_shares.filter(pl.col("window") == red_sea_name[0])
        .select("origin_group", "share").iter_rows()
    )
    gear_losers = [o for o in gear_base_rs if gear_base_rs[o] > gear_resp_rs.get(o, 0.0)]
    gear_gainers = [o for o in gear_base_rs if gear_resp_rs.get(o, 0.0) > gear_base_rs.get(o, 0.0)]
    for loser in gear_losers[:2]:
        for gainer in gear_gainers[:2]:
            key_pairs.append(("gear (Red Sea)", loser, gainer, gear_base_rs, gear_resp_rs))

summary_rows = []
for case_label, src, tgt, b, r in key_pairs:
    p = pairwise_diversion_rates(b, r).get((src, tgt))
    q = proportional_diversion_rates(b, r).get((src, tgt))
    if p is not None and q is not None:
        summary_rows.append({
            "case": case_label, "from": src, "to": tgt,
            "pairwise": p, "proportional": q,
        })

summary = pl.DataFrame(summary_rows)
if summary.height > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    labels = [f"{r['case']}\n{r['from']}→{r['to']}" for r in summary.to_dicts()]
    x = range(len(labels))
    ax.bar([i - 0.2 for i in x], summary["pairwise"].to_list(), 0.35,
           label="pairwise", color="#c98a2b", edgecolor="#8a6a1b")
    ax.bar([i + 0.2 for i in x], summary["proportional"].to_list(), 0.35,
           label="proportional", color="#2d5b4f", edgecolor="#1a3d34")
    ax.set_xticks(list(x), labels, fontsize=8)
    ax.set_ylabel("diversion rate")
    ax.set_title("Pairwise vs proportional diversion rates")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

## 8. Findings

In [ ]:
findings = [
    f"The pairwise formula gives MEX → TWN = {mex_twn_pair:.4f}; the proportional rule "
    f"gives {mex_twn_prop:.4f}, a {(mex_twn_prop / mex_twn_pair - 1) * 100:+.1f}% change.",
    "The proportional rule attributes each gainer's gain across all losers in proportion to "
    "their losses, so the attributed gains sum to the actual gain by construction; the "
    "pairwise formula over-attributes (each loser gets the full gain independently).",
    "For servers, the difference is modest because Mexico dominates the losses: its share of "
    "the total loss is large, so the proportional weight is close to 1.",
    "The proportional rule changes the magnitudes but not the qualitative finding: Taiwan "
    "gained and Mexico lost the most, under both formulas.",
]
for i, f in enumerate(findings, 1):
    print(f"{i}. {f}")